# pplx-decider-flash vs d1-3B — routing and guardrail decisions on a free T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/pplx-decider-flash-v1.2-vs-d1.ipynb)

Side-by-side **inference** of two open decision models on the same small set of intents:

| Model | Backbone | Licence | Card size | T4 load |
|---|---|---|---|---|
| [`perplexity-ai/pplx-decider-flash-v1.2`](https://huggingface.co/perplexity-ai/pplx-decider-flash-v1.2) | Qwen3.5-4B (~4.54B), distilled from the 27B deciders | Apache 2.0 | ~9.1 GB bf16 | **4-bit NF4 (~3.5 GB)**; float16 (~9.5 GB) if 4-bit fails |
| [`LiquidAI/d1-3B`](https://huggingface.co/LiquidAI/d1-3B) | LFM2.5-VL-3B (3.12B) | [LFM Open License v1.0](https://huggingface.co/LiquidAI/d1-3B/blob/main/LICENSE) | ~6.2 GB | **float16 (~6.2 GB)** |

Neither model writes text. You pass a **state** (text or JSON) and named questions (`noul` yes/no, `choice`, `score`). One forward pass reads calibrated probabilities off a decision head — zero output tokens. That is the same shape the [d1 notebook](d1-3B-inference-decisions.ipynb) uses, so this notebook can ask both models the same questions.

**What this notebook is for:** LLM-as-judge / router style decisions — support triage, an agent guardrail on a proposed tool call, and priority scores.

**What this notebook is not:**

- Not a training run. It does **not** install Unsloth, `FastDecisionModel`, or `DecisionTrainer`, and it does not clone [`qwen3_5_4b_decision_email_triage.ipynb`](qwen3_5_4b_decision_email_triage.ipynb) or [`gemma4_e4b_decision_email_triage.ipynb`](gemma4_e4b_decision_email_triage.ipynb). Those notebooks *train* a decision head on the support-email dataset.
- Not a replacement for [`d1-3B-inference-decisions.ipynb`](d1-3B-inference-decisions.ipynb), which stays with d1-3B (including `system_one_batch` and a longer vision walkthrough).

**Runtime:** *Runtime > Change runtime type > T4 GPU*. A T4 has ~16 GB, no native bfloat16, and no FlashAttention-2. The official pplx loader hardcodes CUDA **bfloat16**; this notebook patches that to **4-bit + SDPA** (eager if SDPA refuses the hybrid Qwen3.5 stack). d1-3B loads in **float16 + SDPA**, same as its own notebook. The two checkpoints do **not** stay resident together: run pplx, free it, then load d1.

**Links:** [pplx-decider-flash-v1.2](https://huggingface.co/perplexity-ai/pplx-decider-flash-v1.2) · [d1-3B](https://huggingface.co/LiquidAI/d1-3B) · [d1 blog](https://www.liquid.ai/blog/d1-open) · [pplx demo](https://huggingface.co/spaces/hugging-apps/pplx-decider-flash)


In [ ]:
%pip install -q -U "transformers>=5.17.0,<6" bitsandbytes accelerate safetensors huggingface_hub pillow pandas

# pplx-decider-flash needs transformers>=5.17 (checkpoint saved with 5.17.0) and Qwen3_5Model.
# d1-3B needs transformers>=5.14 and trust_remote_code. One install covers both.
# bitsandbytes is the T4 4-bit path for the 4.54B flash decider (~9.5 GB in float16).
#
# Do NOT `pip install` the snapshot's ./source package: its pyproject pins torch==2.14.0
# and would replace Colab's CUDA wheel. The model card's first example puts source/src
# on sys.path instead — that is what the load cell does.
#
# Do NOT install unsloth, unsloth_zoo, flash-attn, or flash-linear-attention.
# This notebook is inference-only and uses the reference (padded) Decider path, not the fast path.


> **Restart the runtime now:** *Runtime > Restart session*, then continue from the next cell.
> Colab has already imported its preinstalled transformers. The upgrade is invisible until the process restarts.


In [ ]:
import gc

import torch
import transformers
from packaging.version import Version

assert Version(transformers.__version__) >= Version("5.17.0"), (
    f"transformers {transformers.__version__} is loaded; pplx-decider-flash needs 5.17+. "
    "Run the install cell, then Runtime > Restart session."
)

if not torch.cuda.is_available():
    raise SystemExit("Select Runtime > Change runtime type > T4 GPU, then rerun from the top.")

# T4 (sm_75) has no FlashAttention-2. Keep PyTorch SDPA on the mem-efficient and math kernels.
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)

DEVICE = "cuda"
major, minor = torch.cuda.get_device_capability()
NATIVE_BF16 = torch.cuda.is_bf16_supported(including_emulation=False)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3

# Official pplx Decider uses bfloat16 on CUDA. T4 cannot do that. float16 weights are ~9.5 GB
# on a ~15 GB card (tight once activations land), so default to 4-bit NF4 on pre-Ampere GPUs.
# Ampere+ with 20 GB+ follows the card (bf16, no quant).
if NATIVE_BF16 and VRAM_GB >= 20:
    COMPUTE_DTYPE = torch.bfloat16
    USE_4BIT = False
else:
    COMPUTE_DTYPE = torch.float16
    USE_4BIT = True

print("transformers", transformers.__version__)
print("torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(), f"sm_{major}{minor}", "| compute dtype:", COMPUTE_DTYPE)
print(f"VRAM total: {VRAM_GB:.1f} GB | native bf16: {NATIVE_BF16} | pplx 4-bit: {USE_4BIT}")
!nvidia-smi -L


## 1. Shared prompt suite

Both models speak the same decision schema (`noul` / `choice` / `score`). The cases below are the overlap with [`d1-3B-inference-decisions.ipynb`](d1-3B-inference-decisions.ipynb): support tickets, a JSON tool-call guardrail, and a couple of priority scores. One COCO image is included because **both** checkpoints are multimodal; the image is skipped if the download fails. The comparison stays on **joint** requests (every question about a state in one sequence), which is how d1 is meant to be used. The flash card says one question per request is the safer default for *this* 4B student — a later cell shows that split for one ticket, without changing the table.

| `type` | `criteria` | answer |
|---|---|---|
| `noul` | optional `{"true": ..., "false": ...}` | `noul` = P(yes) |
| `choice` | `{name: description}` | `choice`, `confidence`, `probabilities` |
| `score` | 2+ levels, lowest first | `score` (expected level), `confidence`, `probabilities`, `legend` |

API mapping: pplx `decider.decide({"state", "questions", "images"?})` · d1 `model.system_one(state, questions, images=...)`.


In [ ]:
from copy import deepcopy

TRIAGE = {
    "refund": {
        "type": "noul",
        "instructions": "Is the customer asking for a refund?",
    },
    "team": {
        "type": "choice",
        "instructions": "Which team should handle this?",
        "criteria": {
            "billing": "Charges, refunds, invoices",
            "technical": "App or site faults",
            "fraud": "Suspected unauthorised use",
            "shipping": "Deliveries, tracking, returns",
            "account": "Login, password, profile settings",
        },
    },
    "urgency": {
        "type": "score",
        "instructions": "How urgent is this?",
        "criteria": ["Can wait", "Today", "Blocking the customer now"],
    },
}

GUARDRAIL = {
    "destructive": {
        "type": "noul",
        "instructions": "Does the command delete or overwrite data?",
    },
    "in_scope": {
        "type": "noul",
        "instructions": "Does the command stay within what the user asked for?",
        "criteria": {"true": "Only touches what the user asked about", "false": "Touches anything else"},
    },
    "verdict": {
        "type": "choice",
        "instructions": "What should the agent do with this tool call?",
        "criteria": {
            "allow": "Safe and matches the request",
            "confirm": "Plausible but risky; ask the user first",
            "block": "Deletes or changes things the user did not ask about",
        },
    },
}

PRIORITY = {
    "priority": {
        "type": "score",
        "instructions": "Rate how urgently a human should see this.",
        "criteria": ["no action needed", "this week", "today", "immediately"],
    }
}

VISION = {
    "cats": {
        "type": "choice",
        "instructions": "How many cats are there?",
        "criteria": {"one": "One", "two": "Two", "more": "Three or more"},
    },
    "remote": {
        "type": "noul",
        "instructions": "Is there a remote control in the picture?",
    },
}

CASES = [
    {
        "id": "refund_double_charge",
        "kind": "support_triage",
        "state": "I was charged twice this month, please refund one of them.",
        "questions": deepcopy(TRIAGE),
    },
    {
        "id": "app_crash",
        "kind": "support_triage",
        "state": "The app crashes when I open settings.",
        "questions": deepcopy(TRIAGE),
    },
    {
        "id": "fraud_charges",
        "kind": "support_triage",
        "state": "There are three purchases on my card I never made.",
        "questions": deepcopy(TRIAGE),
    },
    {
        "id": "parcel_late",
        "kind": "support_triage",
        "state": "Where is my parcel? It was due Monday.",
        "questions": deepcopy(TRIAGE),
    },
    {
        "id": "invoice_pdf",
        "kind": "support_triage",
        "state": "Can I get last month's invoice as a PDF?",
        "questions": deepcopy(TRIAGE),
    },
    {
        "id": "login_fr",
        "kind": "support_triage",
        "state": "Je n'arrive pas à me connecter depuis la mise à jour.",
        "questions": deepcopy(TRIAGE),
    },
    {
        "id": "guardrail_rm_etc",
        "kind": "agent_guardrail",
        "state": {
            "user_request": "Clean up the old log files in /var/log/myapp",
            "tool": "shell",
            "command": "rm -rf /var/log/myapp /etc/myapp",
        },
        "questions": deepcopy(GUARDRAIL),
    },
    {
        "id": "guardrail_ls",
        "kind": "agent_guardrail",
        "state": {
            "user_request": "List the files in /var/log/myapp",
            "tool": "shell",
            "command": "ls /var/log/myapp",
        },
        "questions": deepcopy(GUARDRAIL),
    },
    {
        "id": "priority_payments_down",
        "kind": "priority",
        "state": "All production payments are failing with 500s for the last 20 minutes.",
        "questions": deepcopy(PRIORITY),
    },
    {
        "id": "priority_dark_mode",
        "kind": "priority",
        "state": "It would be nice if the settings page had a dark mode toggle.",
        "questions": deepcopy(PRIORITY),
    },
]


def try_load_coco_image():
    # Same COCO val image as the d1 notebook. Skip the vision case if the URL fails.
    try:
        from transformers.image_utils import load_image

        image = load_image("http://images.cocodataset.org/val2017/000000039769.jpg")
        image.thumbnail((512, 512))
        return image
    except Exception as exc:
        print("Skipping vision case; image download failed:", exc)
        return None


coco = try_load_coco_image()
if coco is not None:
    CASES.append(
        {
            "id": "coco_cats",
            "kind": "vision",
            "state": None,  # d1 image-only state
            "pplx_state": "Look at the image.",  # pplx schema requires a state field
            "image": coco,
            "questions": deepcopy(VISION),
        }
    )
    print("vision case: COCO 000000039769 (two cats on a sofa), thumbnail 512 px")
else:
    print("vision case: skipped")

print(f"{len(CASES)} cases:", ", ".join(c["id"] for c in CASES))


In [ ]:
def summarize_answer(answer):
    kind = answer["type"]
    if kind == "noul":
        p = float(answer["noul"])
        return {
            "decision": f"P(yes)={p:.3f}",
            "label": "yes" if p >= 0.5 else "no",
            "confidence": p if p >= 0.5 else 1.0 - p,
            "detail": f"{p:.3f}",
        }
    if kind == "choice":
        probs = answer.get("probabilities") or {}
        compact = " ".join(f"{k}={v:.2f}" for k, v in probs.items())
        return {
            "decision": answer["choice"],
            "label": answer["choice"],
            "confidence": float(answer.get("confidence", 0.0)),
            "detail": compact,
        }
    legend = answer.get("legend") or {}
    score = float(answer["score"])
    level = str(round(score))
    label = legend.get(level, level)
    return {
        "decision": f"{score:.2f} → {label}",
        "label": label,
        "confidence": float(answer.get("confidence", 0.0)),
        "detail": f"{score:.3f}",
    }


def flatten(model_name, case, response):
    rows = []
    for qid, answer in response["answers"].items():
        row = summarize_answer(answer)
        row.update(
            {
                "model": model_name,
                "case": case["id"],
                "kind": case["kind"],
                "question": qid,
                "type": answer["type"],
            }
        )
        rows.append(row)
    return rows


def free_cuda(*names):
    # Drop named globals and return VRAM to the driver before the next checkpoint.
    for name in names:
        obj = globals().pop(name, None)
        if obj is None:
            continue
        for attr in ("backbone", "readout", "fast", "processor", "encoder"):
            if hasattr(obj, attr):
                try:
                    setattr(obj, attr, None)
                except Exception:
                    pass
        del obj
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()


def vram_line(prefix=""):
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    peak = torch.cuda.max_memory_allocated() / 1024**3
    print(f"{prefix}VRAM allocated {allocated:.2f} GB | reserved {reserved:.2f} GB | peak {peak:.2f} GB")


## 2. Load pplx-decider-flash-v1.2

The model card's loader is `Decider.from_pretrained`. On CUDA it always constructs the `Qwen3_5Model` backbone in **bfloat16** and uses `attn_implementation="sdpa"`. That is correct on L4/A100 and wrong on a T4.

The cell below downloads the snapshot (about 9.1 GB the first time), puts `source/src` on `sys.path` (same as the card), and monkey-patches only the backbone `from_pretrained` so T4 gets 4-bit NF4 + SDPA. The official `decide()` API is unchanged. `fast=False`: the packed path needs `flash-linear-attention`, which this notebook does not install.


In [ ]:
import sys
from pathlib import Path

from huggingface_hub import snapshot_download
from transformers import BitsAndBytesConfig

PPLX_ID = "perplexity-ai/pplx-decider-flash-v1.2"
# Short sequences only. The card default max_length is 262,144; training used ~32k.
PPLX_MAX_LENGTH = 8192
# 1.6 MP is the card default and uses ~1,600 image tokens. Cap well below that on a T4.
PPLX_MAX_PIXELS = 262_144

print("Downloading", PPLX_ID, "(~9.1 GB of weights the first time)…")
PPLX_CKPT = Path(snapshot_download(PPLX_ID))
src = PPLX_CKPT / "source" / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

from pplx_decider import Decider
import pplx_decider.model as pplx_model

_ORIG_Qwen_from_pretrained = pplx_model.Qwen3_5Model.from_pretrained


def _patch_qwen_loader(use_4bit, attn_impl):
    def _from_pretrained(pretrained_model_name_or_path, *args, **kwargs):
        kwargs["attn_implementation"] = attn_impl
        if use_4bit:
            kwargs.pop("dtype", None)
            kwargs["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=COMPUTE_DTYPE,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True,
            )
            kwargs.setdefault("device_map", {"": "cuda:0"})
            kwargs.setdefault("low_cpu_mem_usage", True)
        else:
            kwargs["dtype"] = COMPUTE_DTYPE
        model = _ORIG_Qwen_from_pretrained(pretrained_model_name_or_path, *args, **kwargs)
        if use_4bit:
            # Decider.__init__ calls backbone.to(device). bitsandbytes rejects that.
            model.to = lambda *a, **k: model
        return model

    pplx_model.Qwen3_5Model.from_pretrained = staticmethod(_from_pretrained)


def load_pplx(use_4bit, attn_impl="sdpa"):
    _patch_qwen_loader(use_4bit, attn_impl)
    decider = Decider.from_pretrained(
        PPLX_CKPT,
        device=DEVICE,
        fast=False,
        max_length=PPLX_MAX_LENGTH,
        max_pixels=PPLX_MAX_PIXELS,
    )
    decider.readout.to(device=DEVICE, dtype=COMPUTE_DTYPE)
    return decider


pplx = None
pplx_load = {"four_bit": USE_4BIT, "attn": "sdpa"}
for four_bit, attn in (
    (USE_4BIT, "sdpa"),
    (USE_4BIT, "eager"),
    (False, "sdpa"),
    (False, "eager"),
):
    if four_bit == pplx_load["four_bit"] and attn == pplx_load["attn"] and pplx is not None:
        continue
    try:
        print(f"Loading pplx-decider-flash ({'4-bit NF4' if four_bit else COMPUTE_DTYPE}, {attn})…")
        pplx = load_pplx(four_bit, attn)
        pplx_load = {"four_bit": four_bit, "attn": attn}
        break
    except Exception as exc:
        print("  failed:", type(exc).__name__, exc)
        free_cuda("pplx")
        pplx = None

if pplx is None:
    raise RuntimeError("Could not load pplx-decider-flash-v1.2 on this GPU.")

print(
    "pplx ready | 4-bit:" if pplx_load["four_bit"] else "pplx ready | dtype:",
    COMPUTE_DTYPE if not pplx_load["four_bit"] else "NF4",
    "| attn:",
    pplx_load["attn"],
    "| temperature:",
    round(pplx.temperature, 3),
)
vram_line("pplx ")


### Card-style smoke test

The model card's own request (integration 401 → team / blocked / frustration). Confirms `decide()` works before the suite.


In [ ]:
import json

smoke = pplx.decide(
    {
        "state": "My integration keeps failing with a 401 since we rotated our API key. Please help.",
        "questions": {
            "team": {
                "type": "choice",
                "instructions": "Which team should handle this request?",
                "criteria": {
                    "billing": "Charges and refunds",
                    "support": "Technical integration errors",
                    "sales": "Questions about buying a product",
                },
            },
            "urgent": {"type": "noul", "instructions": "Is the customer blocked right now?"},
            "frustration": {"type": "score", "criteria": ["calm", "mildly annoyed", "frustrated", "angry"]},
        },
    }
)
print(json.dumps(smoke, indent=2))
vram_line("after smoke ")


## 3. Run the shared suite on pplx-decider-flash


In [ ]:
PPLX_ROWS = []
PPLX_RESPONSES = {}

for case in CASES:
    request = {
        "state": case.get("pplx_state", case["state"]),
        "questions": case["questions"],
    }
    if case.get("image") is not None:
        request["images"] = [case["image"]]
    try:
        response = pplx.decide(request)
    except Exception as exc:
        print(f"{case['id']:>24}: SKIP {type(exc).__name__}: {exc}")
        continue
    PPLX_RESPONSES[case["id"]] = response
    PPLX_ROWS.extend(flatten("pplx-decider-flash", case, response))
    bits = ", ".join(
        f"{r['question']}={r['decision']} ({r['confidence']:.2f})"
        for r in PPLX_ROWS
        if r["case"] == case["id"]
    )
    print(f"{case['id']:>24}: {bits}")

vram_line("pplx suite ")
print(f"{len(PPLX_ROWS)} answers")


### Optional: joint vs one-question-per-request (pplx only)

The flash card is more accurate on most multi-question tasks when each question is its own request. The comparison table stays joint (fair vs d1). This cell only shows the difference on the double-charge ticket.


In [ ]:
joint_case = next(c for c in CASES if c["id"] == "refund_double_charge")
if joint_case["id"] not in PPLX_RESPONSES:
    print("Skipping joint-vs-split: joint refund case did not run.")
else:
    split_requests = [
        {"state": joint_case["state"], "questions": {qid: q}}
        for qid, q in joint_case["questions"].items()
    ]
    split_answers = {}
    for request in split_requests:
        split_answers.update(pplx.decide(request)["answers"])

    print(f"{'question':<10} {'joint':<28} {'one-per-request':<28}")
    for qid in joint_case["questions"]:
        j = summarize_answer(PPLX_RESPONSES[joint_case["id"]]["answers"][qid])
        s = summarize_answer(split_answers[qid])
        print(f"{qid:<10} {j['decision']:<28} {s['decision']:<28}")


## 4. Free pplx, then load d1-3B

float16 pplx (~9.5 GB) plus float16 d1 (~6.2 GB) does not fit a 16 GB T4 at once. 4-bit pplx might leave room, but the notebook still unloads first so the comparison path is the same on every GPU. d1 is skipped if fewer than 8 GB remain after the free (should not happen on a T4).


In [ ]:
from transformers import AutoModel

free_cuda("pplx")
vram_line("after free pplx ")

D1_ID = "LiquidAI/d1-3B"
D1_ROWS = []
D1_RESPONSES = {}
d1 = None

free_gb = VRAM_GB - torch.cuda.memory_allocated() / 1024**3
if free_gb < 8:
    print(f"Skipping d1-3B: only {free_gb:.1f} GB free after unloading pplx (need ~8 GB).")
else:
    print("Loading", D1_ID, f"in {COMPUTE_DTYPE}…")
    try:
        d1 = AutoModel.from_pretrained(D1_ID, trust_remote_code=True, dtype=COMPUTE_DTYPE).to(DEVICE)
    except Exception as exc:
        print("d1-3B load failed:", type(exc).__name__, exc)
        d1 = None
    if d1 is not None:
        print("d1 parameters:", f"{sum(p.numel() for p in d1.parameters()) / 1e9:.2f}B", "| dtype:", d1.dtype)
        vram_line("d1 ")

        for case in CASES:
            kwargs = {}
            if case.get("image") is not None:
                kwargs["images"] = [case["image"]]
            try:
                response = d1.system_one(case["state"], case["questions"], **kwargs)
            except Exception as exc:
                print(f"{case['id']:>24}: SKIP {type(exc).__name__}: {exc}")
                continue
            D1_RESPONSES[case["id"]] = response
            D1_ROWS.extend(flatten("d1-3B", case, response))
            bits = ", ".join(
                f"{r['question']}={r['decision']} ({r['confidence']:.2f})"
                for r in D1_ROWS
                if r["case"] == case["id"]
            )
            print(f"{case['id']:>24}: {bits}")

        vram_line("d1 suite ")
        print(f"{len(D1_ROWS)} answers")


## 5. Comparison table

One row per question. **Agree** is a coarse label, not an accuracy score: same `choice`; `noul` on the same side of 0.5; `score` rounded to the same level. Confidence is the model's own calibration (`noul` uses max(p, 1−p)).


In [ ]:
from collections import defaultdict

import pandas as pd
from IPython.display import display

by_key = defaultdict(dict)
for row in PPLX_ROWS + D1_ROWS:
    by_key[(row["case"], row["question"])][row["model"]] = row

table = []
for (case_id, qid), models in by_key.items():
    case = next(c for c in CASES if c["id"] == case_id)
    pplx_row = models.get("pplx-decider-flash")
    d1_row = models.get("d1-3B")
    agree = ""
    if pplx_row and d1_row:
        agree = "yes" if pplx_row["label"] == d1_row["label"] else "no"
    table.append(
        {
            "kind": case["kind"],
            "case": case_id,
            "question": qid,
            "pplx": pplx_row["decision"] if pplx_row else "—",
            "pplx_conf": None if not pplx_row else round(pplx_row["confidence"], 3),
            "d1": d1_row["decision"] if d1_row else "—",
            "d1_conf": None if not d1_row else round(d1_row["confidence"], 3),
            "agree": agree,
        }
    )

df = pd.DataFrame(table)
display(df)

if df["agree"].eq("yes").any() or df["agree"].eq("no").any():
    n = df["agree"].isin(["yes", "no"]).sum()
    print(f"agreement: {(df['agree'] == 'yes').sum()} / {n} questions")

if coco is not None:
    display(coco)


## Caveats

- **Licences.** pplx-decider-flash-v1.2 is [Apache 2.0](https://huggingface.co/perplexity-ai/pplx-decider-flash-v1.2). d1-3B is [LFM Open License v1.0](https://huggingface.co/LiquidAI/d1-3B/blob/main/LICENSE), not Apache.
- **Dtype / VRAM.** The T4 has no native bfloat16 and no FlashAttention-2. pplx is 4-bit NF4 + SDPA (float16 + SDPA/eager if bitsandbytes refuses the hybrid Qwen3.5 stack). The official `Decider` class hardcodes CUDA bfloat16 — that patch is T4-only and is not the evaluated card path. d1-3B is float16, matching [`d1-3B-inference-decisions.ipynb`](d1-3B-inference-decisions.ipynb). Figures (~3.5 GB 4-bit, ~9.5 GB fp16, ~6.2 GB d1) are from parameter counts, not a measured Colab run; the cells print real `torch` allocated / reserved / peak.
- **Sequential load.** Do not keep both checkpoints on a 16 GB card. This notebook frees pplx before loading d1.
- **Not the email-triage trainers.** [`qwen3_5_4b_decision_email_triage.ipynb`](qwen3_5_4b_decision_email_triage.ipynb) and [`gemma4_e4b_decision_email_triage.ipynb`](gemma4_e4b_decision_email_triage.ipynb) fine-tune Unsloth `FastDecisionModel` heads on the shared support-email dataset (L4 / 24 GB class). This notebook never trains and never imports Unsloth. pplx-flash happens to be a Qwen3.5-4B finetune, but it is a finished Perplexity decision checkpoint, not that training recipe.
- **Not d1 alone.** For batch routing, CUDA graphs, and a longer vision walkthrough, use [`d1-3B-inference-decisions.ipynb`](d1-3B-inference-decisions.ipynb).
- **Joint vs split.** Flash is distilled to match the 27B teachers' full distributions. On this checkpoint, **one question per request** wins on most Decision Index multi-question tasks; grouping helps only tightly coupled facets. The table uses joint requests so d1 and pplx see the same input. The optional cell above shows the split on one ticket.
- **Calibration.** Flash applies a fitted temperature (1.554 in `decision_config.json`) once, then softmaxes over valid options only. d1 has its own readout calibration. Confidence numbers are not on a shared scale.
- **Custom code.** pplx ships `pplx_decider` next to the weights (`trust` the snapshot you just downloaded). d1 loads with `trust_remote_code=True`. Full-attention layers in pplx are **noncausal** over the request; default causal generate() does not reproduce this checkpoint.
- **Images.** Both models accept images. This notebook keeps one small COCO frame so the comparison stays on routing / guardrails / scores. Drop `max_pixels` further, or skip the vision case, if VRAM spikes.
- **This is not a benchmark.** Ten hand-written states. Agreement is not accuracy.

**Next steps:** swap in your labels; send independent candidates (tools, documents) as separate pplx requests; compile d1 with `model.compile(mode="reduce-overhead")` on a larger NVIDIA GPU (see the [d1 speed notes](https://huggingface.co/LiquidAI/d1-3B#⚡-speed)).
